In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2
import base64
import numpy as np
import time
import re
import datetime
import os

from io import BytesIO
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import load_model

from IPython.display import display, HTML, Javascript
from google.colab import output

In [ ]:
BASE_DIR    = '/content/drive/MyDrive/vcc26_split622'
MODEL_PATH  = BASE_DIR + '/h5/vcc26_vgg16_FE2FineT.keras'
LABELS_PATH = BASE_DIR + '/labels.txt'

model = load_model(MODEL_PATH)
print("Model loaded")

input_shape = model.input_shape
IMG_HEIGHT  = input_shape[1]
IMG_WIDTH   = input_shape[2]
print("Input size:", IMG_WIDTH, "x", IMG_HEIGHT)

with open(LABELS_PATH, 'r') as f:
    class_labels = [line.strip() for line in f.readlines()]

print("Classes loaded:", len(class_labels))

# Warm up the model so first real prediction is not slow
dummy_input = np.zeros((1, IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.float32)
_ = model(dummy_input, training=False)
print("Model warmed up and ready")

Model loaded
Input size: 224 x 224
Classes loaded: 25
Model warmed up and ready


In [ ]:
is_processing  = False
previous_label = ""

In [ ]:
def set_processing_status(status):
    global is_processing, previous_label
    is_processing  = bool(status)
    previous_label = ""
    print("Processing status set to:", is_processing)


def speak(text):
    global previous_label, is_processing
    if not is_processing:
        return

    cleaned = re.sub(r"^\d+\s*", "", text).strip()
    if cleaned == previous_label:
        return
    previous_label = cleaned

    # Only speak if enough time has passed, cancel any queued speech first
    js = (
        'var now = Date.now();'
        'if (now - lastSpeakTime > speakCooldown) {'
        '    window.speechSynthesis.cancel();'
        f'   var msg = new SpeechSynthesisUtterance("{cleaned}");'
        '    msg.rate = 1.1;'
        '    window.speechSynthesis.speak(msg);'
        '    lastSpeakTime = now;'
        '}'
    )
    display(Javascript(js))


def process_frame(frame_data):
    global is_processing, model, class_labels, IMG_HEIGHT, IMG_WIDTH

    if not is_processing:
        return

    try:
        # Decode image from browser
        raw         = frame_data.split(',')[1]
        img_bytes   = base64.b64decode(raw)
        img         = Image.open(BytesIO(img_bytes)).convert('RGB')

        # Resize
        img_resized = img.resize((IMG_WIDTH, IMG_HEIGHT), Image.BILINEAR)

        # Preprocess
        arr         = np.array(img_resized).astype(np.float32) / 255.0
        arr_input   = np.expand_dims(arr, axis=0)

        # Direct model call is faster than model.predict()
        preds       = model(arr_input, training=False).numpy()[0]

        top_index   = np.argmax(preds)
        top_label   = class_labels[top_index]
        top_conf    = preds[top_index]

        # Top 3
        top3        = np.argsort(preds)[-3:][::-1]
        top3_str    = "  |  ".join([
            f"{class_labels[i]}: {preds[i]:.2f}"
            for i in top3
        ])

        conf_str    = f"{top_conf * 100:.1f}%"

        # Speak the result
        speak(top_label)

        # Update UI
        safe_label  = top_label.replace("'", "")
        safe_top3   = top3_str.replace("'", "")
        display(Javascript(
            f"updatePrediction('{safe_label}', '{safe_top3}', '{conf_str}');"
        ))

    except Exception as e:
        print("Error in process_frame:", e)


def save_frame_to_file(frame_data):
    try:
        raw       = frame_data.split(',')[1]
        img_bytes = base64.b64decode(raw)
        img       = Image.open(BytesIO(img_bytes))

        save_dir  = BASE_DIR + '/imgs/'
        os.makedirs(save_dir, exist_ok=True)

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename  = save_dir + f"captured_{timestamp}.png"
        img.save(filename)
        print("Image saved:", filename)

        display(Javascript(
            f"document.getElementById('status').textContent = 'Saved: {filename}';"
        ))
    except Exception as e:
        print("Error saving image:", e)

In [ ]:
output.register_callback('process_frame',          process_frame)
output.register_callback('save_frame_to_file',     save_frame_to_file)
output.register_callback('set_processing_status',  set_processing_status)

print("Callbacks registered successfully")

Callbacks registered successfully


In [ ]:
html_code = """
<div style="font-family: monospace;">

    <video id="webcam" width="320" height="240"
           autoplay playsinline style="display:none;"></video>

    <canvas id="outputCanvas" width="320" height="240"
            style="border:2px solid #333; display:block; margin-bottom:10px;">
    </canvas>

    <div style="margin-bottom:10px;">
        <button id="startBtn"
                style="padding:10px 20px; background:#4CAF50;
                       color:white; border:none; margin-right:10px;">
            Start Webcam
        </button>
        <button id="stopBtn"
                style="padding:10px 20px; background:#f44336;
                       color:white; border:none; margin-right:10px;">
            Stop Webcam
        </button>
        <button id="saveBtn"
                style="padding:10px 20px; background:#FFC107;
                       color:white; border:none;">
            Save Image
        </button>
    </div>

    <div id="status"
         style="padding:8px; background:#f0f0f0; margin-bottom:6px;">
        Status: Not started
    </div>
    <div id="predictionOutput"
         style="padding:8px; background:#ADD8E6;
                font-size:22px; font-weight:bold; margin-bottom:6px;">
        Prediction: N/A
    </div>
    <div id="top3Output"
         style="padding:8px; background:#ADD8E6; margin-bottom:6px;">
        Top 3: N/A
    </div>
    <div id="confidenceOutput"
         style="padding:8px; background:#d4edda;">
        Confidence: N/A
    </div>

</div>

<script>
    var webcamStream    = null;
    var captureInterval = null;
    var FPS             = 1;
    var lastSpeakTime   = 0;
    var speakCooldown   = 2000;

    var video     = document.getElementById('webcam');
    var canvas    = document.getElementById('outputCanvas');
    var ctx       = canvas.getContext('2d');
    var statusDiv = document.getElementById('status');
    var predDiv   = document.getElementById('predictionOutput');
    var top3Div   = document.getElementById('top3Output');
    var confDiv   = document.getElementById('confidenceOutput');

    document.getElementById('startBtn').addEventListener('click', startWebcam);
    document.getElementById('stopBtn').addEventListener('click',  stopWebcam);
    document.getElementById('saveBtn').addEventListener('click',  saveImage);

    function startWebcam() {
        statusDiv.textContent = 'Status: Requesting camera...';
        navigator.mediaDevices.getUserMedia({
            video: { width: 320, height: 240 },
            audio: false
        })
        .then(function(stream) {
            webcamStream          = stream;
            video.srcObject       = stream;
            statusDiv.textContent = 'Status: Running';
            captureInterval       = setInterval(captureFrame, 1000 / FPS);
            google.colab.kernel.invokeFunction(
                'set_processing_status', [true], {}
            );
        })
        .catch(function(err) {
            statusDiv.textContent = 'Camera error: ' + err.message;
        });
    }

    function stopWebcam() {
        if (captureInterval) {
            clearInterval(captureInterval);
            captureInterval = null;
        }
        if (webcamStream) {
            webcamStream.getTracks().forEach(function(t) { t.stop(); });
            webcamStream = null;
        }
        window.speechSynthesis.cancel();
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        statusDiv.textContent = 'Status: Stopped';
        predDiv.textContent   = 'Prediction: N/A';
        top3Div.textContent   = 'Top 3: N/A';
        confDiv.textContent   = 'Confidence: N/A';
        google.colab.kernel.invokeFunction(
            'set_processing_status', [false], {}
        );
    }

    function captureFrame() {
        if (video.srcObject && video.readyState >= 2) {
            ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
            // Lower jpeg quality = smaller data = faster transfer to Python
            var imageData = canvas.toDataURL('image/jpeg', 0.5);
            google.colab.kernel.invokeFunction(
                'process_frame', [imageData], {}
            );
        }
    }

    function saveImage() {
        var imageData = canvas.toDataURL('image/png');
        google.colab.kernel.invokeFunction(
            'save_frame_to_file', [imageData], {}
        );
    }

    function updatePrediction(label, top3_str, confidence) {
        predDiv.textContent = 'Prediction: ' + label;
        top3Div.textContent = 'Top 3: '      + top3_str;
        confDiv.textContent = 'Confidence: ' + confidence;
    }

    window.updatePrediction = updatePrediction;
</script>
"""

display(HTML(html_code))

Processing status set to: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Processing status set to: False
